<a href="https://colab.research.google.com/github/farahnda/Learning/blob/main/Ridge_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RIDGE REGRESSION: TEKNIK ANALISIS DATA YANG ALAMI MULTIKOLINEARITAS & OVERFITTING**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('housing.csv')
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    int64  
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    int64  
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  MEDV     506 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 55.5 KB


In [ ]:
X = df.drop(['MEDV'], axis =1) # x buang kolom MEDV, axis=1 = kolom; axis=0 = baris biar belajar
y = df['MEDV'] # y ambil hanya kolom MEDV

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42) # split jadi 2, testing=25%; training 75%

In [ ]:
# RIDGE REGRESSION
Ridge_model = Ridge(alpha=1).fit(X_train, y_train) # buat model; penalti = besar alpha; 0=kecil; 100=besar sekali; belajar dari data training,
Ridge_model.intercept_ # intercept = konstanta dengan persamaan regresi

np.float64(24.878370472969348)

In [ ]:
y_pred = Ridge_model.predict(X_test) # model nebak harga rumah pada 127 data testing (25%)
np.sqrt(mean_squared_error(y_test, y_pred))# hitung selisih,lalu selisih dikuadratkan

np.float64(4.741357980709097)

In [ ]:
Ridge_model.coef_ # HASILNYA = BOBOT/WEIGHT SETIAP FITUR (angka di coef_ = nilai b1, b2, b3. etc)
# + = RM bertambah, MEDV cenderung naik; - = NOX naik, harga rumah cenderung turun

array([-0.12383039,  0.03139178,  0.01767668,  2.54277179, -8.77249222,
        4.37980204, -0.01537349, -1.29086084,  0.24406848, -0.01082435,
       -0.83346553,  0.01348642, -0.53435396])

In [ ]:
r2_score(y_test, y_pred)  # UKUR SEBERAPA BAIK MODEL JELASKAN VARIASI DATA (1 paling baik sampai 0 atau minus)

0.6789748327846081

In [ ]:
from sklearn.model_selection import GridSearchCV
# CROSS VALIDATION
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1) # data dibagi 10 part; proses diulang 3x; random_state biar ga berubah nilainya

# define grid
grid = dict()
grid['alpha'] = np.arange(0, 1, 0.1) # model coba beberapa nilai alpha karna gatau alpha paling baik apa
model = Ridge() # buat model ridge
search = GridSearchCV(model, grid, scoring='neg_mean_absolute_error', cv = cv, n_jobs = -1) # lakukan pencarian otomatis
# tentuin alpha(0, 1, 0.1) terus cv trus hitung error (pilih alpha dgn error terkecil)

results = search.fit(X_train, y_train) # lakukin proses pencarian
print('MAE: %.3f', results.best_score_) # rata2 prediksi salah
print('Config: %s', results.best_params_) # dari semua alpha, paling bagus = 0.7

MAE: %.3f -3.500259191225439
Config: %s {'alpha': np.float64(0.7000000000000001)}


In [ ]:
Ridge_model = Ridge(alpha = 0.7,).fit(X_train, y_train) # modelnya pake alpha terbaik
y_pred = Ridge_model.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))
# 2 di atas itung lagi, hasilnya jadi lebih kecil karna error lebih kecil = lebih baik

np.float64(4.731437210393682)

In [ ]:
r2_score(y_test, y_pred) # itung lagi, lebih gede, model lebih bagus

0.6803168470441052

In [ ]:
pd.Series(Ridge_model.coef_, index = X_train.columns) # buat coef_ jadi lebih dibaca

,0
CRIM,-0.124643
ZN,0.031027
INDUS,0.023550
CHAS,2.595725
NOX,-10.178221
RM,4.382223
AGE,-0.014263
DIS,-1.311503
RAD,0.246439
TAX,-0.010654
